# 03 — Inventory Intelligence

## Objective

The goal of this notebook is to transform demand forecasts into
inventory management decisions.

We will use the final forecasting model from Notebook 2 to estimate
future demand and combine those forecasts with inventory information
to identify:

- Stockout risk
- Overstock risk
- Safety stock requirements
- Reorder points
- Recommended inventory actions

## 1. Inventory Data Requirements

Before building the inventory intelligence layer, we need to understand
what information is already available in our sales dataset and what
additional inventory information is required.

In [12]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\opsell\EcomAI-OS


In [4]:
from src.data.loader import load_sales_data

df = load_sales_data("../data/raw/sales.csv")

In [5]:
df.head()

,date,product_id,product_name,category,price,discount,promotion,day_of_week,month,is_weekend,units_sold
0,2024-01-01,P001,Wireless Headphones,Electronics,1999.0,0,0,0,1,False,33
1,2024-01-02,P001,Wireless Headphones,Electronics,1999.0,0,0,1,1,False,39
2,2024-01-03,P001,Wireless Headphones,Electronics,1999.0,0,0,2,1,False,39
3,2024-01-04,P001,Wireless Headphones,Electronics,1799.1,10,1,3,1,False,59
4,2024-01-05,P001,Wireless Headphones,Electronics,1999.0,0,0,4,1,False,35


In [6]:
df.shape

(3655, 11)

In [7]:
df.columns.tolist()

['date',
 'product_id',
 'product_name',
 'category',
 'price',
 'discount',
 'promotion',
 'day_of_week',
 'month',
 'is_weekend',
 'units_sold']

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3655 entries, 0 to 3654
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          3655 non-null   str    
 1   product_id    3655 non-null   str    
 2   product_name  3655 non-null   str    
 3   category      3655 non-null   str    
 4   price         3655 non-null   float64
 5   discount      3655 non-null   int64  
 6   promotion     3655 non-null   int64  
 7   day_of_week   3655 non-null   int64  
 8   month         3655 non-null   int64  
 9   is_weekend    3655 non-null   bool   
 10  units_sold    3655 non-null   int64  
dtypes: bool(1), float64(1), int64(5), str(4)
memory usage: 289.2 KB


In [9]:
df.describe()

,price,discount,promotion,day_of_week,month,units_sold
count,3655.000000,3655.000000,3655.000000,3655.000000,3655.000000,3655.000000
mean,1887.947196,5.526676,0.340356,2.991792,6.519836,38.059918
std,682.374279,6.733271,0.473894,2.000941,3.450023,14.283852
min,799.200000,0.000000,0.000000,0.000000,1.000000,12.000000
25%,1349.100000,0.000000,0.000000,1.000000,4.000000,27.000000
50%,1899.050000,0.000000,0.000000,3.000000,7.000000,36.000000
75%,2499.000000,10.000000,1.000000,5.000000,10.000000,46.000000
max,2999.000000,20.000000,1.000000,6.000000,12.000000,116.000000


In [10]:
df["product_id"].nunique()

5

In [11]:
df.groupby(
    ["product_id", "product_name"]
)["units_sold"].agg(
    ["mean", "std", "min", "max"]
).round(2)

,,mean,std,min,max
product_id,product_name,,,,
P001,Wireless Headphones,45.01,12.96,22,105
P002,Running Shoes,31.75,8.70,16,74
P003,Smart Watch,38.19,10.92,19,76
P004,Travel Backpack,25.21,7.00,12,52
P005,Yoga Mat,50.14,14.48,20,116


## 3. Inventory Data Preparation

We create a separate inventory dataset containing the current
inventory state of each product.

The inventory dataset is intentionally kept separate from the
historical sales dataset because sales represent historical events,
while inventory represents the current operational state.

In [13]:
inventory_data = pd.DataFrame({
    "product_id": ["P001", "P002", "P003", "P004", "P005"],
    "lead_time_days": [4, 7, 5, 3, 6],
    "current_stock": [220, 120, 210, 400, 150],
    "unit_cost": [1000, 1800, 2500, 1200, 700],
    "reorder_quantity": [200, 250, 200, 150, 300]
})

inventory_data

,product_id,lead_time_days,current_stock,unit_cost,reorder_quantity
0,P001,4,220,1000,200
1,P002,7,120,1800,250
2,P003,5,210,2500,200
3,P004,3,400,1200,150
4,P005,6,150,700,300
